# MALTO Text Authorship Detection
## Best Model: `stacking_lgbm` (LB 0.92422)

**6-class classification**: Human (0), DeepSeek (1), Grok (2), Claude (3), Gemini (4), ChatGPT (5)  
**Metric**: Macro F1  
**Architecture**: FeatureUnion (7 TF-IDF branches + 49 stylometric features) → StackingClassifier (TwoStage + MLP + LGBM → LR meta)

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightgbm>=4.0', 'xgboost>=2.0', 'scikit-learn>=1.3',
                'numpy', 'pandas', 'scipy'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import math
import re
import string
import unicodedata
import itertools
import warnings
from pathlib import Path
from typing import Dict, List, Union

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
print('Imports OK.')

## 1. Data Paths

In [ ]:
import os

# Detect environment: Kaggle vs local
if os.path.exists('/kaggle/input'):
    # Kaggle environment — adjust dataset name if needed
    INPUT_DIR = Path('/kaggle/input/malto-hackathon')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    # Local environment
    INPUT_DIR = Path('data')
    OUTPUT_DIR = Path('artifacts/submissions')

TRAIN_FILE  = INPUT_DIR / 'train.csv'
TEST_FILE   = INPUT_DIR / 'test.csv'
SAMPLE_FILE = INPUT_DIR / 'sample_submission.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Train : {TRAIN_FILE}')
print(f'Test  : {TEST_FILE}')

## 2. Preprocessing

In [ ]:
class Preprocessor(BaseEstimator, TransformerMixin):
    """
    Minimal text preprocessing that PRESERVES stylistic signals.
    Philosophy: LLMs have distinctive punctuation, capitalization, and whitespace.
    Over-cleaning destroys discriminative features.
    """
    def __init__(self, normalize_unicode=True, strip_whitespace=True,
                 remove_repeated_spaces=True, lowercase=False,
                 remove_punctuation=False, remove_numbers=False):
        self.normalize_unicode = normalize_unicode
        self.strip_whitespace = strip_whitespace
        self.remove_repeated_spaces = remove_repeated_spaces
        self.lowercase = lowercase
        self.remove_punctuation = remove_punctuation
        self.remove_numbers = remove_numbers

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return [self._clean(text) for text in X]

    def _clean(self, text):
        if not isinstance(text, str):
            text = str(text) if text is not None else ''
        if self.normalize_unicode:
            text = unicodedata.normalize('NFC', text)
        if self.strip_whitespace:
            text = text.strip()
        if self.remove_repeated_spaces:
            text = re.sub(r' {2,}', ' ', text)
            text = re.sub(r'\r\n?', '\n', text)
            text = re.sub(r'\t+', ' ', text)
            text = re.sub(r' {2,}', ' ', text)
        if self.lowercase:
            text = text.lower()
        if self.remove_punctuation:
            text = re.sub(r'[^\w\s]', '', text)
        if self.remove_numbers:
            text = re.sub(r'\d+', '<NUM>', text)
        return text

print('Preprocessor defined.')

## 3. Feature Engineering

In [ ]:
# ── Function-word vocabulary ───────────────────────────────────────────────────
FUNCTION_WORDS = [
    'the', 'a', 'an', 'this', 'that', 'these', 'those',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'of', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'from', 'about',
    'as', 'into', 'through', 'during', 'before', 'after', 'above', 'below',
    'between', 'among', 'under', 'over', 'within', 'without', 'upon',
    'and', 'but', 'or', 'so', 'yet',
    'although', 'because', 'since', 'while', 'if', 'unless', 'until',
    'when', 'where', 'whether', 'though',
    'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
    'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'us', 'them',
    'however', 'therefore', 'moreover', 'furthermore', 'additionally',
    'nevertheless', 'nonetheless', 'thus', 'hence', 'consequently',
    'accordingly', 'whereas', 'meanwhile', 'subsequently', 'overall',
    'also', 'too', 'either', 'neither',
    'very', 'quite', 'rather', 'somewhat', 'even', 'just', 'only',
    'indeed', 'certainly', 'particularly', 'especially', 'generally',
    'actually', 'basically', 'literally', 'simply',
    'all', 'some', 'any', 'each', 'every', 'many', 'much', 'few', 'little',
    'more', 'most', 'other', 'another', 'same', 'different', 'various', 'several',
    'which', 'who', 'whom', 'whose', 'what', 'how', 'why',
    'not', 'no', 'never', 'nor',
]


class StyleometricTransformer(BaseEstimator, TransformerMixin):
    """49 hand-crafted stylometric features."""
    _NUM_RE  = re.compile(r'^\s*\d+[\.)].+', re.MULTILINE)
    _HEAD_RE = re.compile(r'^#{1,6}\s', re.MULTILINE)
    _BOLD_RE = re.compile(r'\*\*')
    _ITAL_RE = re.compile(r'(?<!\*)\*(?!\*)')
    _CODE_RE = re.compile(r'`')
    _LINK_RE = re.compile(r'\[.+?\]\(.+?\)')
    _ELIP_RE = re.compile(r'\.\.\.')
    _CAPS_RE = re.compile(r'\b[A-Z]{2,}\b')
    _TRANS_RE = re.compile(
        r'\b(however|therefore|moreover|furthermore|additionally|consequently|'
        r'nevertheless|nonetheless|whereas|thus|hence|accordingly|subsequently|'
        r'in conclusion|in summary|for example|for instance|in particular)\b',
        re.IGNORECASE
    )
    _HEDGE_RE = re.compile(
        r'\b(may|might|could|possibly|perhaps|likely|suggests|suggest|'
        r'appears|appear|seems|seem|potentially|presumably|arguably|'
        r'apparently|typically|usually|often)\b',
        re.IGNORECASE
    )
    _DEF_RE  = re.compile(r'^[A-Z][^.!?\n]{0,80}\b(is|are|was|were)\b', re.IGNORECASE)
    _YEAR_RE = re.compile(r'\b(1[0-9]{3}|20[0-2][0-9])\b')

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return np.array([self._f(t) for t in X], dtype=np.float32)

    def _f(self, text):
        if not text or not isinstance(text, str):
            return [0.0] * 49
        NL2 = '\n\n'
        NL1 = '\n'
        ch = len(text)
        words = text.split(); nw = max(len(words), 1)
        sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        ns = max(len(sents), 1)
        paras = [p.strip() for p in text.split(NL2) if p.strip()]
        np_ = max(len(paras), 1)
        lines = text.split(NL1); nl_ = max(len(lines), 1)

        wl = [len(w.strip(string.punctuation)) for w in words if w.strip(string.punctuation)]
        awl = float(np.mean(wl)) if wl else 0.0
        ttr = len(set(w.lower().strip(string.punctuation) for w in words)) / nw
        lwr = sum(1 for l in wl if l > 6) / nw
        swc = [len(s.split()) for s in sents]
        vss = sum(1 for c in swc if c < 5) / ns
        vls = sum(1 for c in swc if c > 30) / ns
        apc = float(np.mean([len(p) for p in paras])) if paras else 0.0

        cm  = text.count(',') / nw
        per = text.count('.') / nw
        ex  = text.count('!') / nw
        qu  = text.count('?') / nw
        co  = text.count(':') / nw
        se  = text.count(';') / nw
        qt  = (text.count('"') + text.count("'") +
               text.count('\u201c') + text.count('\u201d')) / nw
        pa  = (text.count('(') + text.count(')')) / nw
        el  = len(self._ELIP_RE.findall(text)) / nw
        da  = (text.count('-') + text.count('\u2014')) / nw

        alpha = [c for c in text if c.isalpha()]
        na    = max(len(alpha), 1)
        ucr   = sum(1 for c in alpha if c.isupper()) / na
        dr    = sum(1 for c in text if c.isdigit()) / max(ch, 1)
        cwr   = len(self._CAPS_RE.findall(text)) / nw
        nlr   = text.count(NL1) / max(ch, 1)

        br  = sum(1 for l in lines if l.lstrip().startswith(('-', '*', '\u2022'))) / nl_
        nr  = len(self._NUM_RE.findall(text)) / nl_
        hr  = len(self._HEAD_RE.findall(text)) / nl_
        cr  = len(self._CODE_RE.findall(text)) / max(ch, 1) * 100
        bor = len(self._BOLD_RE.findall(text)) / max(ch, 1) * 100
        ir  = len(self._ITAL_RE.findall(text)) / max(ch, 1) * 100
        lkr = len(self._LINK_RE.findall(text)) / nw
        isr = sum(1 for s in sents if re.match(r'^I\s', s.strip())) / ns

        punct_chars   = [c for c in text if c in string.punctuation]
        punct_variety = len(set(punct_chars)) / max(len(punct_chars), 1)
        if len(swc) > 1:
            sent_cv = float(np.std(swc) / max(np.mean(swc), 1))
        else:
            sent_cv = 0.0
        trans_rate = len(self._TRANS_RE.findall(text)) / ns
        fsw = float(len(sents[0].split())) if sents else 0.0
        cap_words = sum(1 for w in words if w and w[0].isupper())
        proper_noun_dens = max(cap_words - ns, 0) / nw
        hedge_rate = len(self._HEDGE_RE.findall(text)) / ns
        qps = text.count('?') / ns
        sent_range = float(max(swc) - min(swc)) if len(swc) > 1 else 0.0
        text_len_log = math.log10(max(ch, 1))
        starts_with_the = 1.0 if text.startswith('The ') else 0.0
        clause_per_sent = text.count(',') / ns
        def_opening = 1.0 if (sents and self._DEF_RE.match(sents[0].strip())) else 0.0
        art_count = sum(1 for w in words if w.lower() in ('the', 'a', 'an'))
        article_rate = art_count / nw
        cop_count = sum(1 for w in words if w.lower() in ('is', 'are', 'was', 'were'))
        copula_rate = cop_count / ns
        year_rate = len(self._YEAR_RE.findall(text)) / ns

        return [ch, nw, ns, np_, awl, nw/ns, nw/np_, ttr, lwr,
                vss, vls, apc, cm, per, ex, qu, co, se, qt, pa, el, da,
                ucr, dr, cwr, nlr, br, nr, hr, cr, bor, ir, lkr, isr,
                punct_variety, sent_cv, trans_rate, fsw, proper_noun_dens,
                hedge_rate, qps, sent_range, text_len_log,
                starts_with_the, clause_per_sent,
                def_opening, article_rate, copula_rate, year_rate]


class DenseToSparse(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return sp.csr_matrix(X if isinstance(X, np.ndarray) else np.array(X))


class StyleometricPipeline(BaseEstimator, TransformerMixin):
    """Extractor + MaxAbsScaler + sparse conversion. 49 features."""
    def __init__(self):
        self.extractor = StyleometricTransformer()
        self.scaler    = MaxAbsScaler()

    def fit(self, X, y=None):
        self.scaler.fit(self.extractor.transform(X))
        return self

    def transform(self, X, y=None):
        return sp.csr_matrix(self.scaler.transform(self.extractor.transform(X)))


class FunctionWordAnalyzer:
    """Callable analyzer: n-grams of function words only."""
    def __init__(self, ngram_range=(1, 2)):
        self.ngram_range = ngram_range
        self._fw_set = frozenset(FUNCTION_WORDS)
        self._tok_re = re.compile(r'(?u)\b\w+\b')

    def __call__(self, text):
        tokens = [t for t in self._tok_re.findall(text.lower()) if t in self._fw_set]
        min_n, max_n = self.ngram_range
        grams = []
        for n in range(min_n, max_n + 1):
            for i in range(len(tokens) - n + 1):
                grams.append(' '.join(tokens[i:i + n]))
        return grams


class DSGrokSubspaceTfidf(BaseEstimator, TransformerMixin):
    """Word TF-IDF fitted ONLY on DeepSeek (1) + Grok (2) samples."""
    _DEEPSEEK = 1
    _GROK = 2

    def __init__(self, max_features=10000, ngram_range=(1, 3), min_df=1):
        self.max_features = max_features
        self.ngram_range  = ngram_range
        self.min_df       = min_df

    def fit(self, X, y=None):
        if y is not None:
            y_arr = np.array(y)
            mask  = (y_arr == self._DEEPSEEK) | (y_arr == self._GROK)
            X_dg  = [X[i] for i in range(len(X)) if mask[i]]
        else:
            X_dg = list(X)
        if len(X_dg) < 4:
            X_dg = list(X)
        self.tfidf_ = TfidfVectorizer(
            analyzer='word', ngram_range=self.ngram_range,
            max_features=self.max_features, min_df=self.min_df,
            sublinear_tf=True, token_pattern=r'(?u)\b\w+\b',
        )
        self.tfidf_.fit(X_dg)
        return self

    def transform(self, X, y=None):
        return self.tfidf_.transform(X)


class DelexTfidfVectorizer(BaseEstimator, TransformerMixin):
    """Word TF-IDF with digit sequences replaced by __NUM__."""
    _NUM_RE = re.compile(r'\b\d+\b')

    def __init__(self, max_features=30000, ngram_range=(1, 2), min_df=2):
        self.max_features = max_features
        self.ngram_range  = ngram_range
        self.min_df       = min_df

    def _delex(self, text):
        return self._NUM_RE.sub('__NUM__', text)

    def fit(self, X, y=None):
        self.tfidf_ = TfidfVectorizer(
            analyzer='word', ngram_range=self.ngram_range,
            max_features=self.max_features, min_df=self.min_df,
            sublinear_tf=True, token_pattern=r'(?u)\b\w+\b',
        )
        self.tfidf_.fit([self._delex(t) for t in X])
        return self

    def transform(self, X, y=None):
        return self.tfidf_.transform([self._delex(t) for t in X])


def build_feature_union():
    """Assemble 7-branch FeatureUnion (~165k features total)."""
    trans = [
        ('word_tfidf', TfidfVectorizer(
            analyzer='word', ngram_range=(1, 2), max_features=50000,
            min_df=3, max_df=0.95, sublinear_tf=True,
            strip_accents='unicode', token_pattern=r'(?u)\b\w+\b',
        )),
        ('char_tfidf', TfidfVectorizer(
            analyzer='char_wb', ngram_range=(2, 6), max_features=50000,
            min_df=2, max_df=0.95, sublinear_tf=True,
        )),
        ('char_tfidf_micro', TfidfVectorizer(
            analyzer='char', ngram_range=(3, 7), max_features=20000,
            min_df=2, max_df=0.95, sublinear_tf=True,
        )),
        ('stylometric', StyleometricPipeline()),
        ('function_word_tfidf', TfidfVectorizer(
            analyzer=FunctionWordAnalyzer(ngram_range=(1, 3)),
            max_features=5000, min_df=2, sublinear_tf=True,
        )),
        ('ds_grok_tfidf', DSGrokSubspaceTfidf(
            max_features=10000, ngram_range=(1, 3), min_df=1,
        )),
        ('delex_tfidf', DelexTfidfVectorizer(
            max_features=30000, ngram_range=(1, 2), min_df=2,
        )),
    ]
    return FeatureUnion(transformer_list=trans)

print('Feature engineering classes defined.')

## 4. Model Definitions

In [ ]:
_KNOWN_CLASS_DIST = {0: 1520, 1: 80, 2: 160, 3: 80, 4: 240, 5: 320}


class TfidfMLPClassifier(ClassifierMixin, BaseEstimator):
    """TruncatedSVD → MLPClassifier with balanced sample weights."""
    _estimator_type = 'classifier'

    def __init__(self, n_svd_components=500, hidden_layer_sizes=(512, 256),
                 activation='relu', alpha=0.01, max_iter=300,
                 early_stopping=True, validation_fraction=0.1,
                 random_state=42, deepseek_boost_factor=1.0, verbose=False):
        self.n_svd_components = n_svd_components
        self.hidden_layer_sizes = hidden_layer_sizes
        self.activation = activation
        self.alpha = alpha
        self.max_iter = max_iter
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.random_state = random_state
        self.deepseek_boost_factor = deepseek_boost_factor
        self.verbose = verbose

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_components = min(self.n_svd_components, X.shape[1] - 1, X.shape[0] - 1)
        self.svd_ = TruncatedSVD(n_components=n_components, random_state=self.random_state)
        X_dense = self.svd_.fit_transform(X)
        sample_weights = np.asarray(compute_sample_weight('balanced', y), dtype=np.float64)
        if self.deepseek_boost_factor != 1.0:
            sample_weights[np.asarray(y) == 1] *= float(self.deepseek_boost_factor)
        self.mlp_ = MLPClassifier(
            hidden_layer_sizes=self.hidden_layer_sizes, activation=self.activation,
            solver='adam', alpha=self.alpha, max_iter=self.max_iter,
            early_stopping=self.early_stopping,
            validation_fraction=self.validation_fraction,
            random_state=self.random_state, n_iter_no_change=15, verbose=self.verbose,
        )
        self.mlp_.fit(X_dense, y, sample_weight=sample_weights)
        return self

    def predict(self, X):
        return self.mlp_.predict(self.svd_.transform(X))

    def predict_proba(self, X):
        return self.mlp_.predict_proba(self.svd_.transform(X))


class TwoStageClassifier(ClassifierMixin, BaseEstimator):
    """
    Two-stage classifier for the hard DeepSeek/Grok boundary.
    Stage 1: 6-class base. Stage 2: binary DS/Grok specialist.
    """
    _estimator_type = 'classifier'
    DEEPSEEK = 1
    GROK = 2

    def __init__(self, base_classifier=None, binary_classifier=None,
                 margin_trigger_gap=None, binary_ds_threshold=0.5, top2_trigger=False):
        self.base_classifier = base_classifier
        self.binary_classifier = binary_classifier
        self.margin_trigger_gap = margin_trigger_gap
        self.binary_ds_threshold = binary_ds_threshold
        self.top2_trigger = top2_trigger

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.base_clf_ = clone(self.base_classifier) if self.base_classifier else \
            LogisticRegression(C=0.5, max_iter=1000, solver='lbfgs',
                               class_weight='balanced', random_state=42)
        self.base_clf_.fit(X, y)
        mask = (y == self.DEEPSEEK) | (y == self.GROK)
        if mask.sum() < 4:
            self.binary_clf_ = None
            return self
        X_bin, y_bin = X[mask], y[mask]
        self.binary_clf_ = clone(self.binary_classifier) if self.binary_classifier else \
            LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                               class_weight=None, random_state=42)
        self.binary_clf_.fit(X_bin, y_bin)
        return self

    def _compute_trigger_mask(self, X, preds, base_proba=None):
        base_ambig = (preds == self.DEEPSEEK) | (preds == self.GROK)
        if base_proba is None:
            return base_ambig
        conditions = [base_ambig]
        if self.top2_trigger:
            sorted_idx = np.argsort(base_proba, axis=1)[:, -2:]
            top2_are_ds_grok = np.all(np.isin(sorted_idx, [self.DEEPSEEK, self.GROK]), axis=1)
            conditions.append(top2_are_ds_grok)
        if self.margin_trigger_gap is not None:
            margin = np.abs(base_proba[:, self.DEEPSEEK] - base_proba[:, self.GROK])
            conditions.append(margin < self.margin_trigger_gap)
        result = conditions[0]
        for cond in conditions[1:]:
            result = result & cond
        return result

    def _apply_binary(self, X_ambig, bin_proba=None):
        if bin_proba is None:
            if hasattr(self.binary_clf_, 'predict_proba'):
                bin_proba = self.binary_clf_.predict_proba(X_ambig)
            else:
                return self.binary_clf_.predict(X_ambig)
        bin_classes = self.binary_clf_.classes_
        if self.DEEPSEEK not in bin_classes or self.GROK not in bin_classes:
            return self.binary_clf_.predict(X_ambig)
        ds_pos = int(np.where(bin_classes == self.DEEPSEEK)[0][0])
        return np.where(bin_proba[:, ds_pos] >= self.binary_ds_threshold, self.DEEPSEEK, self.GROK)

    def predict(self, X):
        base_proba = None
        need_proba = self.margin_trigger_gap is not None or self.top2_trigger
        if hasattr(self.base_clf_, 'predict_proba') and need_proba:
            base_proba = self.base_clf_.predict_proba(X)
            preds = np.argmax(base_proba, axis=1).copy()
        else:
            preds = self.base_clf_.predict(X).copy()
        if self.binary_clf_ is None:
            return preds
        mask = self._compute_trigger_mask(X, preds, base_proba)
        if mask.any():
            preds[mask] = self._apply_binary(X[mask])
        return preds

    def predict_proba(self, X):
        if not hasattr(self.base_clf_, 'predict_proba'):
            raise AttributeError('Base classifier has no predict_proba')
        proba = self.base_clf_.predict_proba(X).copy()
        if self.binary_clf_ is None or not hasattr(self.binary_clf_, 'predict_proba'):
            return proba
        preds = np.argmax(proba, axis=1)
        mask  = self._compute_trigger_mask(X, preds, proba)
        if mask.any():
            bin_proba   = self.binary_clf_.predict_proba(X[mask])
            bin_classes = self.binary_clf_.classes_
            ds_pos = int(np.where(bin_classes == self.DEEPSEEK)[0][0]) if self.DEEPSEEK in bin_classes else None
            gk_pos = int(np.where(bin_classes == self.GROK)[0][0]) if self.GROK in bin_classes else None
            if ds_pos is not None and gk_pos is not None:
                for i, idx in enumerate(np.where(mask)[0]):
                    total = proba[idx, self.DEEPSEEK] + proba[idx, self.GROK]
                    if total > 0:
                        proba[idx, self.DEEPSEEK] = total * bin_proba[i, ds_pos]
                        proba[idx, self.GROK]     = total * bin_proba[i, gk_pos]
        return proba


class LGBMTfidfClassifier(ClassifierMixin, BaseEstimator):
    """TruncatedSVD + LightGBM gradient boosting."""
    _estimator_type = 'classifier'

    def __init__(self, n_svd_components=300, n_estimators=300, num_leaves=31,
                 learning_rate=0.05, min_child_samples=10, subsample=0.8,
                 colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, random_state=42):
        self.n_svd_components  = n_svd_components
        self.n_estimators      = n_estimators
        self.num_leaves        = num_leaves
        self.learning_rate     = learning_rate
        self.min_child_samples = min_child_samples
        self.subsample         = subsample
        self.colsample_bytree  = colsample_bytree
        self.reg_alpha         = reg_alpha
        self.reg_lambda        = reg_lambda
        self.random_state      = random_state

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.svd_ = TruncatedSVD(n_components=self.n_svd_components, random_state=self.random_state)
        X_dense = self.svd_.fit_transform(X)
        cw = compute_class_weight('balanced', classes=self.classes_, y=y)
        sample_weight = np.array([cw[int(np.where(self.classes_ == yi)[0][0])] for yi in y])
        self.lgbm_ = LGBMClassifier(
            n_estimators=self.n_estimators, num_leaves=self.num_leaves,
            learning_rate=self.learning_rate, min_child_samples=self.min_child_samples,
            subsample=self.subsample, colsample_bytree=self.colsample_bytree,
            reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            random_state=self.random_state, n_jobs=-1, verbose=-1,
        )
        self.lgbm_.fit(X_dense, y, sample_weight=sample_weight)
        return self

    def predict(self, X):
        return self.lgbm_.predict(self.svd_.transform(X))

    def predict_proba(self, X):
        return self.lgbm_.predict_proba(self.svd_.transform(X))


print('Model classes defined.')

## 5. Threshold Optimizer

In [ ]:
def apply_thresholds(proba, thresholds):
    """Scale class probabilities and return argmax."""
    scaled = proba * thresholds
    return np.argmax(scaled, axis=1)


def optimize_thresholds(proba, y_true, n_grid=11, lo=0.5, hi=3.0):
    """Grid-search per-class scale factors to maximise Macro F1."""
    n_classes  = proba.shape[1]
    thresholds = np.ones(n_classes)
    grid       = np.linspace(lo, hi, n_grid)

    baseline_preds = np.argmax(proba, axis=1)
    per_class_f1   = f1_score(y_true, baseline_preds, average=None, zero_division=0)
    baseline_macro = f1_score(y_true, baseline_preds, average='macro', zero_division=0)
    print(f'  Threshold opt: baseline macro F1 = {baseline_macro:.4f}')

    class_order = np.argsort(per_class_f1)  # worst classes first
    best_macro  = baseline_macro

    for _pass in range(2):
        for cls in class_order:
            best_val = thresholds[cls]
            for val in grid:
                thresholds[cls] = val
                preds = apply_thresholds(proba, thresholds)
                macro = f1_score(y_true, preds, average='macro', zero_division=0)
                if macro > best_macro:
                    best_macro = macro
                    best_val   = val
            thresholds[cls] = best_val

    final_preds = apply_thresholds(proba, thresholds)
    final_macro = f1_score(y_true, final_preds, average='macro', zero_division=0)
    print(f'  Threshold opt: tuned  macro F1 = {final_macro:.4f}')
    print(f'  Thresholds: {[round(t, 3) for t in thresholds]}')
    return thresholds


def apply_ds_grok_pair_threshold(proba, preds, pair_threshold):
    """Post-process DS/Grok predictions using ratio threshold."""
    DEEPSEEK_CLASS, GROK_CLASS = 1, 2
    preds = preds.copy()
    ds_sum = proba[:, DEEPSEEK_CLASS] + proba[:, GROK_CLASS]
    ambiguous = (ds_sum > 0.15) & ((preds == DEEPSEEK_CLASS) | (preds == GROK_CLASS))
    if not ambiguous.any():
        return preds
    ratio = proba[ambiguous, DEEPSEEK_CLASS] / np.maximum(ds_sum[ambiguous], 1e-9)
    preds[ambiguous] = np.where(ratio >= pair_threshold, DEEPSEEK_CLASS, GROK_CLASS)
    return preds


def optimize_ds_grok_threshold(proba, y_true, n_grid=21):
    """Grid-search the DS/Grok pair ratio threshold."""
    DEEPSEEK_CLASS, GROK_CLASS = 1, 2
    ds_grok_count = int(((y_true == DEEPSEEK_CLASS) | (y_true == GROK_CLASS)).sum())
    if ds_grok_count < 10:
        return 0.5
    baseline_preds = np.argmax(proba, axis=1)
    best_macro = f1_score(y_true, baseline_preds, average='macro', zero_division=0)
    best_thr = 0.5
    for thr in np.linspace(0.25, 0.75, n_grid):
        preds_c = apply_ds_grok_pair_threshold(proba, baseline_preds, thr)
        macro = f1_score(y_true, preds_c, average='macro', zero_division=0)
        if macro > best_macro:
            best_macro = macro
            best_thr   = thr
    print(f'  DS/Grok pair threshold: {best_thr:.3f} (macro F1 = {best_macro:.4f})')
    return float(best_thr)


print('Threshold optimizer defined.')

## 6. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_FILE)
test_df  = pd.read_csv(TEST_FILE)

print(f'Train shape: {train_df.shape}')
print(f'Test shape : {test_df.shape}')
print(f'Train columns: {train_df.columns.tolist()}')
print(f'Test columns : {test_df.columns.tolist()}')
train_df.head(3)

In [ ]:
# Identify text/label column names
text_col  = 'TEXT'  if 'TEXT'  in train_df.columns else train_df.columns[0]
label_col = 'LABEL' if 'LABEL' in train_df.columns else train_df.columns[1]

X_train = train_df[text_col].fillna('').astype(str).tolist()
y_train = train_df[label_col].values.astype(int)
X_test  = test_df[text_col if text_col in test_df.columns else test_df.columns[0]] \
               .fillna('').astype(str).tolist()

print(f'X_train: {len(X_train)} samples')
print(f'X_test : {len(X_test)} samples')
print(f'Class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}')

## 7. Build & Train Pipeline (stacking_lgbm)

Architecture: `Preprocessor → FeatureUnion(7 branches) → StackingClassifier`

| Base estimator | Description |
|---|---|
| `two_stage_top2` | LR balanced + binary DS/Grok specialist (top-2 trigger) |
| `mlp_svd` | TruncatedSVD(500) + MLP(512,256) — non-linear boundaries |
| `lgbm_svd` | TruncatedSVD(500) + LightGBM — gradient boosting |
| **Meta** | LogisticRegression(C=0.1, balanced) on 18-dim OOF probas |

In [ ]:
SEED = 42

def build_stacking_lgbm():
    """Build the best stacking_lgbm model."""
    # ── Base 1: TwoStage top-2 trigger ───────────────────────────────────────
    top2_base = LogisticRegression(
        C=0.5, max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=SEED
    )
    top2_binary = LogisticRegression(
        C=1.5, max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=SEED
    )
    base_two_stage = TwoStageClassifier(
        base_classifier=top2_base,
        binary_classifier=top2_binary,
        top2_trigger=True,
        margin_trigger_gap=0.40,
        binary_ds_threshold=0.50,
    )

    # ── Base 2: MLP on SVD ────────────────────────────────────────────────────
    base_mlp = TfidfMLPClassifier(
        n_svd_components=500, hidden_layer_sizes=(512, 256), activation='relu',
        alpha=0.01, max_iter=300, early_stopping=True, validation_fraction=0.1,
        random_state=SEED, deepseek_boost_factor=1.0, verbose=False,
    )

    # ── Base 3: LGBM on SVD ───────────────────────────────────────────────────
    base_lgbm = LGBMTfidfClassifier(
        n_svd_components=500, n_estimators=500, num_leaves=63,
        learning_rate=0.05, min_child_samples=10,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=SEED,
    )

    # ── Meta: LR on 18-dim OOF probas (3 models × 6 classes) ─────────────────
    meta_lr = LogisticRegression(
        C=0.1, max_iter=1000, class_weight='balanced', random_state=SEED
    )

    stacking = StackingClassifier(
        estimators=[
            ('two_stage_top2', base_two_stage),
            ('mlp_svd',        base_mlp),
            ('lgbm_svd',       base_lgbm),
        ],
        final_estimator=meta_lr,
        cv=3,
        stack_method='predict_proba',
        n_jobs=1,
        passthrough=False,
    )
    return stacking


# ── Full pipeline ─────────────────────────────────────────────────────────────
preprocessor   = Preprocessor(
    normalize_unicode=True, strip_whitespace=True,
    remove_repeated_spaces=True, lowercase=False,
    remove_punctuation=False, remove_numbers=False,
)
feature_union  = build_feature_union()
stacking_model = build_stacking_lgbm()

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('features',     feature_union),
    ('classifier',   stacking_model),
])

print('Pipeline built.')
print(f'Steps: {[name for name, _ in pipeline.steps]}')

In [ ]:
import time

print(f'Training on {len(X_train)} samples...')
t0 = time.time()
pipeline.fit(X_train, y_train)
print(f'Training complete in {time.time() - t0:.1f}s')

## 8. Threshold Optimization (5-Fold OOF)

In [ ]:
from sklearn.model_selection import StratifiedKFold

print('Running 5-fold OOF for threshold optimization...')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_proba = np.zeros((len(X_train), 6))
X_arr = np.array(X_train)  # for indexing

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
    print(f'  Fold {fold+1}/5...')
    X_tr, X_val = X_arr[tr_idx].tolist(), X_arr[val_idx].tolist()
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    fold_pipeline = Pipeline([
        ('preprocessor', Preprocessor(
            normalize_unicode=True, strip_whitespace=True,
            remove_repeated_spaces=True, lowercase=False,
            remove_punctuation=False, remove_numbers=False,
        )),
        ('features',     build_feature_union()),
        ('classifier',   build_stacking_lgbm()),
    ])
    fold_pipeline.fit(X_tr, y_tr)
    oof_proba[val_idx] = fold_pipeline.predict_proba(X_val)

oof_preds = np.argmax(oof_proba, axis=1)
oof_f1    = f1_score(y_train, oof_preds, average='macro', zero_division=0)
print(f'\nOOF Macro F1 (before threshold): {oof_f1:.4f}')
print(f'Per-class F1: {[round(x, 3) for x in f1_score(y_train, oof_preds, average=None, zero_division=0)]}')

In [ ]:
print('\nOptimizing per-class thresholds...')
thresholds = optimize_thresholds(oof_proba, y_train)

print('\nOptimizing DS/Grok pair threshold...')
ds_grok_threshold = optimize_ds_grok_threshold(oof_proba, y_train)

# Evaluate with thresholds
thresh_preds = apply_thresholds(oof_proba, thresholds)
thresh_preds = apply_ds_grok_pair_threshold(oof_proba, thresh_preds, ds_grok_threshold)
thresh_f1    = f1_score(y_train, thresh_preds, average='macro', zero_division=0)
print(f'\nOOF Macro F1 (after threshold) : {thresh_f1:.4f}')

## 9. Generate Test Predictions

In [ ]:
print(f'Running inference on {len(X_test)} test samples...')
test_proba = pipeline.predict_proba(X_test)

# Apply thresholds
test_preds = apply_thresholds(test_proba, thresholds)
test_preds = apply_ds_grok_pair_threshold(test_proba, test_preds, ds_grok_threshold)

CLASS_NAMES = ['Human', 'DeepSeek', 'Grok', 'Claude', 'Gemini', 'ChatGPT']
unique, counts = np.unique(test_preds, return_counts=True)
print('\nPrediction distribution:')
for cls, cnt in zip(unique, counts):
    print(f'  Class {cls} ({CLASS_NAMES[cls]}): {cnt}')

## 10. Save Submission

In [ ]:
from datetime import datetime

# Build submission dataframe
submission_df = pd.DataFrame({
    'id':    range(len(test_preds)),
    'label': test_preds.astype(int),
})

# Validate against sample submission if available
if SAMPLE_FILE.exists():
    sample = pd.read_csv(SAMPLE_FILE)
    assert len(submission_df) == len(sample), \
        f'Length mismatch: submission={len(submission_df)}, sample={len(sample)}'
    print(f'Submission length matches sample: {len(submission_df)} rows ✓')

# Save
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path  = OUTPUT_DIR / f'submission_{timestamp}.csv'
submission_df.to_csv(out_path, index=False)
print(f'Submission saved to: {out_path}')

submission_df.head(10)

In [ ]:
print('=== SUMMARY ===')
print(f'Model         : stacking_lgbm')
print(f'Features      : 7-branch FeatureUnion (~165k features)')
print(f'Stylometric   : 49 hand-crafted features')
print(f'OOF Macro F1  : {oof_f1:.4f} (pre-threshold)')
print(f'OOF Macro F1  : {thresh_f1:.4f} (post-threshold)')
print(f'Thresholds    : {[round(t, 3) for t in thresholds]}')
print(f'DS/Grok pair  : {ds_grok_threshold:.3f}')
print(f'Test samples  : {len(test_preds)}')
print(f'Submission    : {out_path}')